## FPA module 2: market overview

This module gathers information about multiple companies within a targeted list for monitoring. 
Including:
* moving averages (1M, 6M, 1Y) and comparing to current
* historical max and min and comparing to current
* fundamental factors (margins, growth, DTE ,ROA, ROE, ...)

In [1]:
#importing libraries

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import yfinance as yf
from datetime import datetime, timedelta

# import plotly.express as px
# import plotly.graph_objs as go
# from plotly.subplots import make_subplots
# from dash import Dash, html, dash_table, dcc, callback, Output, Input, 

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
tickers_df = pd.read_csv('C:/Users/luengoag/Downloads/finance_portfolio_dataset.csv', usecols=[0, 1])
tickers_list = tickers_df['index'].tolist()



In [3]:
# Download 2 years of historical data with 5-day intervals
data = yf.download(tickers_list, period='2y', interval='5d')

# valid periods: “1d”, “5d”, “1mo”, “3mo”, “6mo”, “1y”, “2y”, “5y”, “10y”, “ytd”, “max”
# valid intervales: “1m”, “2m”, “5m”, “15m”, “30m”, “60m”, “90m”, “1h”, “1d”, “5d”, “1wk”, “1mo”, “3mo”

close_df = data['Close']

# Slice data for approximate periods
close_1mo = close_df[-4:]    # ~1 month (4 data points)
close_6mo = close_df[-26:]   # ~6 months (26 data points)
close_1y = close_df[-52:]    # ~1 year (52 data points)

C:\Users\luengoag\AppData\Local\Temp\ipykernel_30456\1261438890.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers_list, period='2y', interval='5d')
[*********************100%***********************]  80 of 80 completed

1 Failed download:
['DFS']: YFPricesMissingError('possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")')


In [4]:
# Calculate averages
avg_1mo = close_1mo.mean().rename("Avg_1M")
avg_6mo = close_6mo.mean().rename("Avg_6M")
avg_1y = close_1y.mean().rename("Avg_1Y")

# Latest close price (last row)
latest_close = close_df.iloc[-1].rename("Latest_Close")

# Historical max close price per ticker
historical_max = close_df.max().rename('Hist_Max')

# Historical max close price per ticker
historical_min = close_df.min().rename('Hist_Min')



In [5]:
cols_to_keep = [   
    'country',
    'sector',
    'industryKey',
#    'marketCap',
    'averageDailyVolume3Month',
    'dividendRate',
#    'totalCash',
    'ebitda',
    'quickRatio',
    'currentRatio',
#    'totalRevenue',
    'debtToEquity',
    'returnOnAssets',
    'returnOnEquity',
    'grossProfits', 
#    'freeCashflow', 
#    'operatingCashflow',
    'earningsGrowth',
    'revenueGrowth',
    'grossMargins',
#    'ebitdaMargins',
    'operatingMargins',
    'profitMargins'
]

In [6]:
# Collect info for each ticker
info_list = []
for ticker in tickers_list:
    try:
        data = yf.Ticker(ticker).info
        filtered_info = {col: data.get(col, None) for col in cols_to_keep}
        filtered_info['Ticker'] = ticker
        info_list.append(filtered_info)
    except Exception as e:
        print(f"Error fetching info for {ticker}: {e}")
# Convert to DataFrame
info_df = pd.DataFrame(info_list).set_index('Ticker')

summary = pd.concat([latest_close, avg_1mo, avg_6mo, avg_1y, historical_max, historical_min], axis=1)

# Merge with your existing summary (assumes summary has tickers as index)
summary = summary.join(info_df)

In [7]:
# Calculate percentage differences vs averages and historical max
summary['%_vs_1M_Avg'] = ((summary['Latest_Close'] - summary['Avg_1M']) / summary['Avg_1M'])
summary['%_vs_6M_Avg'] = ((summary['Latest_Close'] - summary['Avg_6M']) / summary['Avg_6M'])
summary['%_vs_1Y_Avg'] = ((summary['Latest_Close'] - summary['Avg_1Y']) / summary['Avg_1Y'])
summary['%_vs_Hist_Max'] = ((summary['Latest_Close'] - summary['Hist_Max']) / summary['Hist_Max'])
summary['%_vs_Hist_Min'] = ((summary['Latest_Close'] - summary['Hist_Min']) / summary['Hist_Min'])

# Calculate recent trend: month-over-month % change (using last 5 data points)
mom_change = close_df.pct_change().rolling(window=4).sum().iloc[-1].rename('MoM_Change')
summary = pd.concat([summary, mom_change], axis=1)

In [8]:
# Define thresholds
far_below_max_threshold = -0.2
near_min_threshold = 0.2

# Classify stocks
def classify(row):
    if (row['%_vs_Hist_Max'] < far_below_max_threshold) and (row['MoM_Change'] > 0):
        return 'Opportunity / Recovery'
    elif (row['%_vs_Hist_Min'] < near_min_threshold) and (row['MoM_Change'] < 0):
        return 'Structural Issues'
    else:
        return 'Neutral'

summary['Status'] = summary.apply(classify, axis=1)

In [9]:
cols = ['country','industryKey', 'Latest_Close',
        'dividendRate','averageDailyVolume3Month',
#        'Avg_1M', 'Avg_6M', 'Avg_1Y', 'Hist_Max', 'Hist_Min', 
        '%_vs_1M_Avg', '%_vs_6M_Avg', '%_vs_1Y_Avg' ,'%_vs_Hist_Max', '%_vs_Hist_Min', 'MoM_Change','Status',
        'quickRatio','debtToEquity','returnOnAssets','returnOnEquity',
        'revenueGrowth','earningsGrowth','grossMargins','operatingMargins','profitMargins']
summary = summary[cols]

# format floats nicely
pd.set_option('display.float_format', '{:.3f}'.format)

In [10]:
summary[summary['Status']!= 'Neutral'].sort_values(by=['earningsGrowth','revenueGrowth'],ascending=[False,False])

,country,industryKey,Latest_Close,dividendRate,averageDailyVolume3Month,%_vs_1M_Avg,%_vs_6M_Avg,%_vs_1Y_Avg,%_vs_Hist_Max,%_vs_Hist_Min,MoM_Change,Status,quickRatio,debtToEquity,returnOnAssets,returnOnEquity,revenueGrowth,earningsGrowth,grossMargins,operatingMargins,profitMargins
Ticker,,,,,,,,,,,,,,,,,,,,,
AMD,United States,semiconductors,159.331,NaN,46674863.000,0.060,0.382,0.224,-0.246,0.862,0.109,Opportunity / Recovery,1.765,8.174,0.026,0.039,0.359,5.252,0.536,0.108,0.080
JD,China,internet-retail,33.455,1.000,11471326.000,0.035,-0.077,-0.029,-0.238,0.636,0.018,Opportunity / Recovery,0.806,28.325,0.041,0.163,0.158,0.582,0.099,0.035,0.038
QCOM,United States,semiconductors,158.680,3.560,8269572.000,0.005,0.027,-0.004,-0.247,0.506,0.019,Opportunity / Recovery,1.838,52.737,0.134,0.422,0.169,0.220,0.557,0.284,0.261
REGN,United States,biotechnology,549.170,3.520,1317604.000,0.007,-0.092,-0.295,-0.540,0.137,0.056,Opportunity / Recovery,3.900,9.204,0.069,0.160,-0.037,0.159,0.488,0.199,0.319
LMT,United States,aerospace-defense,464.205,13.200,1453326.000,-0.003,0.005,-0.066,-0.226,0.213,0.013,Opportunity / Recovery,0.873,303.816,0.083,0.825,0.045,0.139,0.102,0.127,0.077
TGT,United States,discount-stores,103.470,4.500,6806549.000,0.012,-0.027,-0.161,-0.393,0.157,0.070,Opportunity / Recovery,0.152,130.213,0.066,0.291,-0.028,0.118,0.281,0.063,0.040
MRK,United States,drug-manufacturers-general,79.580,3.240,15681286.000,-0.018,-0.049,-0.162,-0.381,0.057,0.001,Opportunity / Recovery,0.827,71.992,0.138,0.393,-0.016,0.075,0.771,0.401,0.273
COP,United States,oil-gas-e-p,91.670,3.120,7938327.000,-0.007,-0.019,-0.077,-0.284,0.074,0.032,Opportunity / Recovery,1.023,36.457,0.088,0.167,0.185,0.037,0.490,0.267,0.161
PG,United States,household-personal-products,155.080,4.230,8483637.000,-0.003,-0.047,-0.056,-0.120,0.137,-0.017,Structural Issues,0.445,64.975,0.110,0.303,-0.021,0.013,0.517,0.233,0.185
